# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets with their `@id`s and fields
print("Available record sets and their fields/columns:")
record_sets = []
for rs in dataset.record_sets():
    print(f'Record set name: {rs.name}')
    print(f'  @id: {rs.id}')
    field_ids = []
    # Get field id & field name for each field in this record set
    for field in rs.fields:
        print(f'    - Field: {field.name}, @id: {field.id}, column: {field.column.id if field.column else None}, dataType: {field.data_type}')
        field_ids.append(field.id)
    record_sets.append({'name': rs.name, 'id': rs.id, 'field_ids': field_ids})
    print()
if not record_sets:
    print('No record sets were found. This may be a metadata-only package or missing data files.')

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

dataframes = {}
if record_sets:
    # Use all available record set @ids
    record_set_ids = [rs['id'] for rs in record_sets]
    print("RecordSet @ids:", record_set_ids)
    for record_set_id in record_set_ids:
        print(f'Loading records for {record_set_id}...')
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f'Columns in DataFrame for {record_set_id}: {df.columns.tolist()}')
            display(df.head())
        else:
            print(f"No records found for record set: {record_set_id}")
else:
    print('No record sets available in this dataset.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming distributions, or grouping by key attributes.

In [ ]:
# This block demonstrates filtering, normalization, and grouping on one record set with numeric fields.
import numpy as np

if dataframes:
    # Pick the first available record set for demonstration
    first_record_set_id = list(dataframes.keys())[0]
    df = dataframes[first_record_set_id]

    print(f'Working with record set: {first_record_set_id}')
    # Identify numeric columns to use for analysis
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f'Numeric columns found: {numeric_cols}')
    if len(numeric_cols) == 0:
        print('No numeric columns found - skipping EDA.')
    else:
        # Use first numeric column
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() > 0 else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        col_norm = f"{numeric_field}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, col_norm]].head())

        # Try grouping by a non-numeric field if available
        candidate_group_cols = [c for c in df.columns if c not in numeric_cols]
        group_field = candidate_group_cols[0] if len(candidate_group_cols) > 0 else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (showing means of numeric columns):")
            display(grouped_df.head())
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Plot a histogram and scatter if appropriate fields exist.
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and len(numeric_cols) > 0:
    # Histogram of the first numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If there are multiple numeric columns, plot scatter
    if len(numeric_cols) > 1:
        plt.figure(figsize=(6,6))
        sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.title(f'Scatter plot: {numeric_cols[0]} vs {numeric_cols[1]}')
        plt.show()
else:
    print('Not enough numeric data for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset structure and schema were loaded from the Croissant JSON-LD.
- Available record sets and their schemas were displayed, and tabular data loaded where possible.
- Sample EDA (filter, normalization, grouping) and visualizations were provided for numeric fields.
- To go further, use domain knowledge to interpret field meanings and extend the analysis.